# Tabular Foundation Models with TabPFN

**PyLadies Amsterdam · online · 20 August 2026 · 18:00 CEST**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/TuanaCelik/tabular-foundation-models-tabpfn-aug2026/blob/main/workshop/tabpfn_workshop.ipynb)

> This is the **participant** notebook. Cells marked `--- YOUR TURN ---` are yours
> to fill in; everything else runs as-is.
> Stuck, or want to read ahead? The filled-in version is in
> [`solutions/tabpfn_workshop_solutions.ipynb`](../solutions/tabpfn_workshop_solutions.ipynb).

### What this workshop is about

You have probably prompted a large language model with a handful of examples and
watched it pick up the pattern without anyone retraining anything. That trick is
called **in-context learning (ICL)**.

**TabPFN** does the same thing, but for tables. You hand it labelled rows as
*context*, and it predicts the label for new rows in a single forward pass.
No gradient descent on your data. No hyperparameter search. No `n_estimators`
grid to tune at 2am.

### What you will do today

| | Section | Task |
|---|---|---|
| | In-context learning for tables | 50 rows of context, one forward pass |
| 1 | **Classification** | Malignant or benign, with honest probabilities |
| 2 | **Regression** | Same two lines of code, but the answer is a number |
| 3 | **Interpretability** | Which features actually drive the prediction? |
| 4 | Forecasting | A time series is just a table with timestamps |
| 5 | Advanced | Thinking mode: spend compute, get a better model |
| | Predictive agents | Where this goes next |

We use **TabPFN-3** through the Prior Labs API, so **you do not need a GPU** —
Colab's free CPU runtime is plenty.

### Datasets

Both are open datasets that ship with scikit-learn, so nothing to download and
nothing sensitive leaves your machine beyond what is already public:

- [**Breast cancer Wisconsin (diagnostic)**](https://scikit-learn.org/stable/datasets/toy_dataset.html#breast-cancer-wisconsin-diagnostic-dataset)
  — 569 rows, 30 features, malignant vs. benign. Our main dataset.
- [**Diabetes**](https://scikit-learn.org/stable/datasets/toy_dataset.html#diabetes-dataset)
  — 442 rows, 10 features, disease progression as a continuous number. Used for regression.

---
## 0. Setup

Below we have 2 options:
1. Use the model locally
2. Use the model via the client (this is our preference for today)

In [ ]:
#Local Installation
!pip install "tabpfn" "tabpfn-extensions[interpretability]"

In [ ]:
#Client Installation
%pip install -q "tabpfn-client" "tabpfn-extensions[interpretability]" shap tabpfn-time-series

### Get your API key

1. [Sign up here](https://ux.priorlabs.ai/?utm_source=workshop&utm_campaign=pyladies)
2. Accept the license for TabPFN-3!
3. Confirm your email and finish account setup.
4. Open the **API Keys** page and copy your key.
5. Paste it into the prompt from the next cell.

The free tier is enough for this whole workshop.

> **Two things to know before you send data to the API**
> - Your key is a secret. `getpass` below keeps it out of the notebook output, so
>   the key never ends up in the file you push to GitHub.
> - Rows you `fit()` on are uploaded to Prior Labs' servers for inference. Today's
>   datasets are public, so that is fine. For confidential data, use the
>   open-source [`tabpfn`](https://github.com/PriorLabs/TabPFN) package on your own
>   hardware — same scikit-learn API, runs locally on a GPU.

In [ ]:
#If you're uing the client
import getpass

import tabpfn_client

# Paste the key you copied from the API Keys page.
tabpfn_client.set_access_token(getpass.getpass("Prior Labs API key: "))

# Alternative: `tabpfn_client.init()` walks you through login/registration
# interactively, right here in the notebook, if you would rather not use the UI.

print(tabpfn_client.get_api_usage())

Next, depending on whethter you're using TabPFN via the client or locally, make sure you have the relevant import line commented out for `TabPFNClassifier` and `TabPFNRegressor`

In [2]:
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.datasets import load_breast_cancer, load_diabetes
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    log_loss,
    r2_score,
    roc_auc_score,
    root_mean_squared_error,
)
from sklearn.model_selection import train_test_split

#from tabpfn_client import TabPFNClassifier, TabPFNRegressor
from tabpfn import TabPFNClassifier, TabPFNRegressor

# TabPFN-3 is deterministic by default: random_state defaults to 0, so the same
# context plus the same query rows give you byte-identical predictions.
RANDOM_STATE = 0

pd.set_option("display.precision", 3)
plt.rcParams["figure.dpi"] = 110

---
## In-context learning, but for tables

If you already have a mental model for few-shot prompting an LLM, you can reuse
almost all of it:

| Few-shot prompting an LLM | TabPFN |
|---|---|
| Examples you paste into the prompt | Labelled **rows** of your table — the *context* |
| The question at the end of the prompt | The row you want a prediction for — the *query* |
| Pretrained on a large text corpus | Pretrained on ~millions of **synthetic** tabular datasets |
| No weight updates when you add examples | No weight updates when you add rows |
| One forward pass | One forward pass |
| Samples tokens; run it twice, get two answers | Returns a calibrated probability distribution; deterministic |

TabPFN keeps scikit-learn's `fit()` / `predict()` names because that is what
every tool in the ecosystem expects — but `fit()` is *not* training. It hands
your rows to the model as context and does the bookkeeping. That is why this
notebook calls them:

```
context_features, context_labels   # the labelled rows the model learns from, in context
query_features,   query_labels     # the rows we want predictions for (+ the truth, for scoring)
```

instead of `X_train, y_train, X_test, y_test`. Nothing is being trained. Same
arrays, honest names.

### Hello, TabPFN

50 rows of context. No feature engineering, no tuning, no imports beyond what we
already have.

In [3]:
cancer = load_breast_cancer(as_frame=True)
features, labels = cancer.data, cancer.target

# The labels arrive as 0/1. This is the mapping from integer to what it means
CLASS_NAMES = {int(i): str(name) for i, name in enumerate(cancer.target_names)}
print(CLASS_NAMES)

context_features, query_features, context_labels, query_labels = train_test_split(
    features,
    labels,
    test_size=0.5,
    stratify=labels,
    random_state=RANDOM_STATE,
)

print(f"context: {context_features.shape[0]} rows x {context_features.shape[1]} features")
print(f"query:   {query_features.shape[0]} rows")
print("labels:")
for label, name in CLASS_NAMES.items():
    print(f"  {label} = {name:<10} ({(labels == label).sum()} rows)")

context_features.head(3)

{0: 'malignant', 1: 'benign'}
context: 284 rows x 30 features
query:   285 rows
labels:
  0 = malignant  (212 rows)
  1 = benign     (357 rows)


,mean radius,mean texture,mean perimeter,mean area,mean smoothness,mean compactness,mean concavity,mean concave points,mean symmetry,mean fractal dimension,radius error,texture error,perimeter error,area error,smoothness error,compactness error,concavity error,concave points error,symmetry error,fractal dimension error,worst radius,worst texture,worst perimeter,worst area,worst smoothness,worst compactness,worst concavity,worst concave points,worst symmetry,worst fractal dimension
226,10.44,15.46,66.62,329.6,0.105,0.077,0.007,0.012,0.179,0.065,0.191,0.903,1.208,11.86,0.007,0.008,0.003,0.005,0.015,0.003,11.52,19.80,73.47,395.4,0.134,0.115,0.026,0.045,0.262,0.083
444,18.03,16.85,117.50,990.0,0.089,0.123,0.109,0.063,0.172,0.058,0.299,0.591,1.921,35.77,0.004,0.016,0.030,0.010,0.013,0.002,20.38,22.02,133.30,1292.0,0.126,0.267,0.429,0.153,0.284,0.082
27,18.61,20.25,122.10,1094.0,0.094,0.107,0.149,0.077,0.170,0.057,0.853,1.849,5.632,93.54,0.011,0.027,0.051,0.019,0.023,0.004,21.31,27.26,139.90,1403.0,0.134,0.212,0.345,0.149,0.234,0.074


In [4]:
context_labels.head(3)

226    1
444    0
27     0
Name: target, dtype: int64

In [5]:
# Just 50 labelled rows as context.
tiny_context_features = context_features.head(50)
tiny_context_labels = context_labels.head(50)

clf_tiny = TabPFNClassifier()
clf_tiny.fit(tiny_context_features, tiny_context_labels)
tiny_preds = clf_tiny.predict(query_features)

print(f"context rows: {len(tiny_context_features)}")

pd.DataFrame({
    "predicted": pd.Series(tiny_preds, index=query_labels.index).map(CLASS_NAMES),
    "actual": query_labels.map(CLASS_NAMES),
    "match": tiny_preds == query_labels.to_numpy(),
})

context rows: 50


,predicted,actual,match
366,malignant,malignant,True
406,benign,benign,True
435,malignant,malignant,True
280,malignant,malignant,True
11,malignant,malignant,True
...,...,...,...
118,malignant,malignant,True
97,benign,benign,True
6,malignant,malignant,True
497,benign,benign,True


---
## 1. Classification: malignant or benign?

### What we are predicting

Someone took a biopsy image of a breast tumour and measured 30 things about the
cells in it: radius, texture, how irregular the outline is, and so on. That is one
row. The label is what the tumour turned out to be: **malignant** or **benign**.


A row of facts in, one of two answers out.

### Two answers, not one

Every classifier can give you two things, and the second is the useful one:

```python
clf.predict(query_features)        # the decision: 0 or 1
clf.predict_proba(query_features)  # the confidence: e.g. 0.97 benign, 0.03 malignant
```

`predict_proba` returns one column per class. Column 1 here is P(benign), because
scikit-learn orders classes `[malignant, benign]` for this dataset.

### Three ways to score it

- **Accuracy** — *what share did I get right?* Easy to read, but easy to fool: if
  95% of your rows are one class, always guessing that class scores 95%.
- **ROC AUC** — *how well do I rank?* The chance the model gives a random benign
  case a higher benign-probability than a random malignant one. **0.5 is a coin
  flip, 1.0 is perfect.** It ignores where you put the decision threshold, which
  makes it the fairest single number for comparing models.
- **Log loss** — *are my probabilities honest?* It punishes confident mistakes
  hard. A model that says "99% benign" about a malignant tumour is penalised far
  more than one that said "60%". **Lower is better.**

Accuracy alone will not tell you whether a model is trustworthy. That is why we
print all three.

### 1a. Fit on the full context

In [ ]:
# --- YOUR TURN ------------------------------------------------------------
# TODO: 1) fit a TabPFNClassifier on context_features / context_labels
#       2) get hard predictions -> `pred`, and the probability of class 1
#          ("benign") -> `proba` (hint: predict_proba returns one column per class)
#       3) print accuracy, ROC AUC and log loss
# your code here

confusion = pd.DataFrame(
    confusion_matrix(query_labels, pred),
    index=[f"actual {name}" for name in CLASS_NAMES.values()],
    columns=[f"predicted {name}" for name in CLASS_NAMES.values()],
)
confusion

Note the **log loss**, not just accuracy. TabPFN's probabilities are calibrated,
which is what you need when a prediction feeds a decision with asymmetric costs —
a missed malignancy is not the same kind of mistake as a false alarm.

In [ ]:
# Where is the model unsure? The rows whose probability sits closest to 0.5.
uncertain = (
    pd.DataFrame({"p_benign": proba, "actual": query_labels.map(CLASS_NAMES)})
    .assign(distance_from_decision_boundary=lambda df: (df["p_benign"] - 0.5).abs())
    .nsmallest(5, "distance_from_decision_boundary")
)
print("The 5 rows TabPFN is least sure about:")
uncertain

### 1b. More context, better predictions

This is the in-context learning story in one plot. We give TabPFN a growing slice
of the same context and score each one — **without retraining anything**. Each
`fit()` is just a different prompt.

**Your turn.** Loop over the context sizes and collect the scores.

In [ ]:
CONTEXT_SIZES = [20, 50, 100, 200, len(context_features)]
records = []

# --- YOUR TURN ------------------------------------------------------------
# TODO: for each n in CONTEXT_SIZES:
#         - take the first n rows of context_features / context_labels
#         - fit a fresh TabPFNClassifier on that slice
#         - predict probabilities for query_features
#         - append {"n_context": n, "model": "TabPFN-3", "roc_auc": ..., "log_loss": ...}
#           to `records`
#       (5 API calls, roughly 20 seconds in total)
# your code here

icl_curve = pd.DataFrame(records)
icl_curve

In [ ]:
fig, ax = plt.subplots(figsize=(5.6, 3.6))

ax.plot(icl_curve["n_context"], icl_curve["roc_auc"], marker="o", linewidth=2.4)
for _, row in icl_curve.iterrows():
    ax.annotate(
        f"{row['roc_auc']:.3f}",
        xy=(row["n_context"], row["roc_auc"]),
        xytext=(0, 7),
        textcoords="offset points",
        ha="center",
        fontsize=8,
    )

ax.set_xscale("log")
ax.set_xticks(CONTEXT_SIZES)
ax.set_xticklabels([str(n) for n in CONTEXT_SIZES])
ax.minorticks_off()
ax.set(xlabel="rows of context", ylabel="ROC AUC", title="More context, better predictions")
ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()


Two things worth saying out loud:

1. **The curve starts high.** A few dozen labelled rows already get you something
   usable — a regime where a model trained from scratch has essentially nothing to
   learn from. That is the pretraining paying off: TabPFN has seen millions of
   synthetic tabular problems and arrives with a prior about what tabular data
   looks like.
2. **It keeps improving, and each point cost one API call**, not a training run. In
   a normal workflow, "let me try it with more data" means retraining. Here it means
   passing a longer context.

---
## 2. Regression: predicting a number

A prediction that's a **Regression** means the answer is a **number** rather than a category — "how much / how many / how
long?" instead of "which one?".

### What we are predicting today

442 people with diabetes each had a check-up. Someone wrote down their age, BMI,
blood pressure, cholesterol, blood sugar — one row per person. A **year later**,
their disease progression was measured on a scale from **25 to 346**, higher meaning
the disease had advanced further.

So: **given the check-up, predict the number that will show up a year from now.**

If you would rather think about it in your own domain, it is the same shape of
problem as:

- how many units of this product will sell next month
- what will this flat sell for
- how many hours will this repair take
- how much will this customer spend this year
- how long until this machine needs servicing

All of them: a row of known facts in, one number out.

### Two ways to score a number-predictor

You need these to read the next cells. Both come from `sklearn.metrics`:

- **RMSE** (root mean squared error) — *how far off am I, typically?* In the same
  units as the thing you are predicting. RMSE of 55 means "usually wrong by about 55
  progression points". **Lower is better**, 0 is perfect. Big misses are punished
  harder than small ones.
- **R²** — *how much of the variation did I explain?* On a 0-to-1 scale.
  **0 means your model is no better than always guessing the average**, 1 means
  perfect. 0.5 means you have explained about half the spread. **Higher is better.**

That is genuinely all you need. We print an "always guess the average" baseline
alongside them so you can see what the numbers mean rather than taking them on faith.

---

Two things about this particular dataset, worth 60 seconds because you will hit both
patterns in your own data:

> **1. It arrives pre-scrambled.** `load_diabetes()` gives you the columns
> **standardised** by default — mean-centred and divided by `std * sqrt(n)` — so a
> 48-year-old shows up as `age = -0.0018`. That is an artefact of how the data was
> shipped with the original 2004 paper. Classical models often
> want scaled inputs; **TabPFN does not care** — raw scales, mixed units and
> categorical columns all go in as they come. So we pass `scaled=False` and work with
> real years and real mm Hg, which also means every plot and importance below is in
> units a human can argue with.

> **2. One column is undocumented.** `sex` is coded `1` and `2` and **nobody says
> which is which** — scikit-learn describes every other column and leaves this one
> blank. The biology hints at an answer: group `1` averages 54 mg/dL HDL, group
> `2` averages 45, and adult reference ranges are 50–60 for women against 40–50 for
> men, so `1` is likely female.. We leave it as `1`/`2` anyway

In [ ]:
# scaled=False gives the raw measurements instead of the standardised ones.
diabetes = load_diabetes(as_frame=True, scaled=False)

READABLE_NAMES = {
    "bp": "blood pressure",
    "s1": "total cholesterol",
    "s2": "ldl",
    "s3": "hdl",
    "s4": "cholesterol / hdl",
    "s5": "log triglycerides",
    "s6": "blood sugar",
}
diabetes_features = diabetes.data.rename(columns=READABLE_NAMES)

reg_context_features, reg_query_features, reg_context_labels, reg_query_labels = train_test_split(diabetes_features,
                                                                                                  diabetes.target,
                                                                                                  test_size=0.2,
                                                                                                  random_state=RANDOM_STATE,
                                                                                                )

print(f"context: {reg_context_features.shape[0]} patients the model gets to learn from")
print(f"query:   {reg_query_features.shape[0]} patients we will predict for")
print(f"target:  {diabetes.target.min():.0f} to {diabetes.target.max():.0f}, "
      f"average {diabetes.target.mean():.0f}")

preview = reg_context_features.head(4).copy()
preview["--> progression in 1 year"] = reg_context_labels.head(4)
preview

### 2a. Your turn: fit and score

**Your turn.** Three lines. Hand TabPFN the context, ask it to predict the query
patients.

```python
reg = TabPFNRegressor()
reg.fit(context_features, context_labels)   # uploads the context — no training
predictions = reg.predict(query_features)   # one forward pass
```

That is the entire API. Same `fit` / `predict` you know from scikit-learn, no
constructor arguments needed — the server picks the latest model (TabPFN-3) for you.

In [ ]:
# --- YOUR TURN ------------------------------------------------------------
# TODO: 1) create a TabPFNRegressor and call it `reg`
#       2) fit it on reg_context_features / reg_context_labels
#       3) predict on reg_query_features -> store in `reg_pred`
# your code here

truth = np.asarray(reg_query_labels)

# The dumbest possible model, for scale: ignore every feature, always guess the
# average of the context. Any real model has to beat this.
always_average = np.full_like(truth, reg_context_labels.mean(), dtype=float)

rmse = root_mean_squared_error(truth, reg_pred)
r2 = r2_score(truth, reg_pred)

print(f"TabPFN-3          RMSE {rmse:6.1f}   R2 {r2:5.2f}")
print(
    f"always guess avg  RMSE {root_mean_squared_error(truth, always_average):6.1f}"
    f"   R2 {r2_score(truth, always_average):5.2f}"
)
print()
print(f"Read that as: typically off by ~{rmse:.0f} points on a 25-346 scale,")
print(f"and {r2:.0%} of the patient-to-patient variation is explained.")

In [ ]:
# One dot per patient. On the dashed line = predicted exactly right. Above it =
# the model was too pessimistic, below = too optimistic.
fig, ax = plt.subplots(figsize=(4.6, 4.6))

ax.scatter(truth, reg_pred, alpha=0.65, edgecolor="none", label="a patient")
lims = [truth.min() - 15, truth.max() + 15]
ax.plot(lims, lims, "--", color="grey", linewidth=1, label="perfect prediction")
ax.axhline(reg_context_labels.mean(), color="C3", linewidth=1, linestyle=":", label="always guess average")

ax.set(
    xlim=lims,
    ylim=lims,
    xlabel="actual progression, one year later",
    ylabel="TabPFN's prediction",
    title=f"RMSE {rmse:.0f}  ·  R² {r2:.2f}",
)
ax.set_aspect("equal")
ax.legend(fontsize=8, loc="upper left")
plt.tight_layout()
plt.show()

---
## 3. Interpretability: what is the model actually using?

"Foundation model" does not have to mean "black box". This section follows the
[Prior Labs cookbook](https://docs.priorlabs.ai/cookbook/interpret_results?utm_source=workshop&utm_campaign=pyladies):

Here, we'll use a dataset called `heart-statlog` which lists various health features per individual, and labels them as to whether thet have heart disease present or not. 

| Your question | View |
|---|---|
| Which features drive predictions **across all patients**? | SHAP **beeswarm** |
| What features contributed to the outcome of **this one** patient? | SHAP **waterfall** |
| How does one feature move the prediction **across its range**? | **partial dependence** |

**SHAP values**: Every prediction starts from an average, and SHAP tells you how much each piece of information moved it away from that average

**`fit_mode="fit_with_cache"`.** Everything here predicts many times against the same
  context; this has the server keep the encoded context in a KV cache so those
  predictions skip re-encoding it.


In [ ]:
import shap

from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split

# The raw column names are cryptic and the categorical values are bare integers, so we
# rename and keep a decoder — worth it, because every plot and printout below is easier
# to read for it. (Same mapping as the cookbook.)
RENAME = {
    "age": "Age",
    "sex": "Sex",
    "chest": "Chest pain type",
    "resting_blood_pressure": "Resting BP",
    "serum_cholestoral": "Cholesterol",
    "fasting_blood_sugar": "High fasting sugar",
    "resting_electrocardiographic_results": "Resting ECG",
    "maximum_heart_rate_achieved": "Max heart rate",
    "exercise_induced_angina": "Exercise angina",
    "oldpeak": "ST depression",
    "slope": "ST slope",
    "number_of_major_vessels": "Blocked vessels",
    "thal": "Thallium scan",
}
DECODE = {
    "Sex": {0: "female", 1: "male"},
    "Chest pain type": {1: "typical angina", 2: "atypical angina", 3: "non-anginal",
                        4: "asymptomatic"},
    "High fasting sugar": {0: "no", 1: "yes"},
    "Resting ECG": {0: "normal", 1: "ST-T abnormality", 2: "LV hypertrophy"},
    "Exercise angina": {0: "no", 1: "yes"},
    "ST slope": {1: "upsloping", 2: "flat", 3: "downsloping"},
    "Thallium scan": {3: "normal", 6: "fixed defect", 7: "reversible defect"},
}
CATEGORICAL = ["Sex", "Chest pain type", "High fasting sugar", "Resting ECG",
               "Exercise angina", "ST slope", "Thallium scan"]
CONTINUOUS = ["Age", "Resting BP", "Cholesterol", "Max heart rate", "ST depression",
              "Blocked vessels"]

heart = fetch_openml("heart-statlog", version=1, as_frame=True)
heart_features = heart.data.astype(float).rename(columns=RENAME)
heart_labels = (heart.target == "present").astype(int)   # 1 = disease present

feature_names = list(heart_features.columns)
categorical_indices = [feature_names.index(c) for c in CATEGORICAL]

X_context, X_query, y_context, y_query = train_test_split(
    heart_features,
    heart_labels,
    train_size=200,
    test_size=70,
    random_state=RANDOM_STATE,
    stratify=heart_labels,
)

# categorical_features_indices: no one-hot encoding, just say which columns they are.
# .values, not the DataFrame — see the note above.
clf_interp = TabPFNClassifier(
    fit_mode="fit_with_cache",
    categorical_features_indices=categorical_indices,
)
clf_interp.fit(X_context.values, y_context.values)

proba_query = clf_interp.predict_proba(X_query.values)[:, 1]
print(f"{len(feature_names)} features ({len(CATEGORICAL)} of them categorical), "
      f"{len(X_context)} context patients")
print(f"ROC AUC on {len(X_query)} held-out patients: {roc_auc_score(y_query, proba_query):.4f}")
print(f"disease present in {y_context.mean():.0%} of the context patients")

X_context.head(3)


In [ ]:
# --- YOUR TURN ------------------------------------------------------------
# TODO: 1) `predict`: a function taking an array of rows, returning P(disease) —
#          clf_interp.predict_proba(np.asarray(rows))[:, 1]
#       2) `background`: 50 sampled context patients, the "average patient" SHAP
#          compares against — shap.sample(X_context, 50, random_state=RANDOM_STATE)
#       3) `explainer`: shap.PermutationExplainer(predict, background)
#       4) `shap_values`: explainer(X_query, max_evals=2 * len(feature_names) + 1)
#          (the slow bit — a couple of minutes)
# your code here

print(f"one SHAP value per feature per patient: {shap_values.values.shape}")
print(f"base rate (average predicted risk): {shap_values.base_values[0]:.0%}")

# display_data colours the dots by the raw feature value, so "high"/"low" mean something.
shap_values.display_data = X_query.values
shap.plots.beeswarm(shap_values, max_display=len(feature_names), show=False)
fig = plt.gcf()
fig.set_size_inches(9, 6)
fig.suptitle(
    "Global drivers of coronary-disease risk — every dot is a patient\n"
    "right = pushed towards disease, red = high feature value",
    fontsize=11,
)
plt.show()


**How to read a beeswarm.** One row per feature, ordered by how much it moves
predictions overall. One dot per patient. Right means that feature pushed *this*
patient towards disease, left means away from it, and the colour is the feature's raw
value — red high, blue low.

### 3b. What caused the prediction for *this* patient?

Same SHAP values, one row at a time. And because the values are already computed,
looking at more patients costs nothing extra.

In [ ]:
# Every patient's prediction = base rate + all of their SHAP pushes.
predicted_risk = shap_values.values.sum(axis=1) + shap_values.base_values
base_rate = float(shap_values.base_values[0])


def readable(feature, value):
    """Show categorical values as words, everything else as a number."""
    if feature in DECODE:
        return DECODE[feature].get(int(value), str(value))
    return f"{value:g}"


def explain_patient(index, label):
    row = shap_values[index]
    values = X_query.iloc[index]

    print(f"[{label}] predicted risk {predicted_risk[index]:.0%}  "
          f"(base rate {base_rate:.0%}, actually "
          f"{'present' if y_query.iloc[index] else 'absent'})")
    for j in np.argsort(-np.abs(row.values))[:5]:
        feature = feature_names[j]
        verb = "raises" if row.values[j] > 0 else "lowers"
        print(f"    {feature:<20} = {readable(feature, values.iloc[j]):<18} {verb} risk by "
              f"{abs(row.values[j]) * 100:4.1f} points")
    print()

    # Words instead of bare numbers on the waterfall's left-hand labels.
    row.display_data = np.array(
        [readable(feature_names[j], values.iloc[j]) for j in range(len(feature_names))],
        dtype=object,
    )
    shap.plots.waterfall(row, show=False)
    fig = plt.gcf()
    fig.set_size_inches(9, 5)
    fig.suptitle(f"{label} — predicted risk {predicted_risk[index]:.0%}", fontsize=11)
    plt.show()


explain_patient(int(np.argmax(predicted_risk)), "Highest-risk patient")


In [ ]:
# Free to look at more: the SHAP values are already computed.
explain_patient(int(np.abs(predicted_risk - 0.5).argmin()), "Borderline patient")


### More for later

**Feature interactions.** `tabpfn_extensions.interpretability.shapiq` with
  `index="k-SII", max_order=2` reports *pairs* — which two features only matter together.
  It also has an imputation-based explainer that reuses the KV cache, which is faster
  than the generic SHAP path we used here.

Documented here:

- [Interpretability capabilities](https://docs.priorlabs.ai/capabilities/interpretability?utm_source=workshop&utm_campaign=pyladies)
- [Cookbook: interpreting results](https://docs.priorlabs.ai/cookbook/interpret_results?utm_source=workshop&utm_campaign=pyladies)

---
## 4. Forecasting with TabPFN-TS

Although TabPFN wasn't trained for time-series forecasting specifically, we _can_ use TabPFN for this use case by treating time-series as a tabular problem.

The entire setup:

```python
pipeline = TabPFNTSPipeline(tabpfn_mode=TabPFNMode.CLIENT)
forecast = pipeline.predict_df(history_df, prediction_length=24)
```

`history_df` needs two columns — `timestamp` and `target` — and you get back a point
forecast plus quantiles. The pipeline does the feature engineering (running index,
calendar features, automatically detected seasonality) and the API calls for you.

Our series: monthly atmospheric CO₂ from Mauna Loa, which ships with `statsmodels`.
Strong trend, strong annual cycle — the two things a forecaster has to get right.

`tabpfn-time-series` came with the install in section 0, so there is nothing new to
set up here.

To _improve_ results, we can do a bit of feature engineering and add additional time-series features/


In [ ]:
from statsmodels.datasets import co2

# The whole input format: one row per timestamp, one column for the value.
series = co2.load_pandas().data["co2"].resample("MS").mean().interpolate()
co2_df = (
    series.loc["1965":]          # skip the sparser early years
    .rename("target")
    .rename_axis("timestamp")
    .reset_index()
)

HORIZON = 24  # months to forecast

history_df = co2_df.iloc[:-HORIZON]   # what the model gets to see
future_df = co2_df.iloc[-HORIZON:]    # held back, only used to score the forecast

print(f"history:  {len(history_df)} months "
      f"({history_df['timestamp'].iloc[0]:%Y-%m} to {history_df['timestamp'].iloc[-1]:%Y-%m})")
print(f"forecast: {len(future_df)} months "
      f"({future_df['timestamp'].iloc[0]:%Y-%m} to {future_df['timestamp'].iloc[-1]:%Y-%m})")
history_df.tail(3)


In [ ]:
from tabpfn_time_series import TabPFNMode, TabPFNTSPipeline

# --- YOUR TURN ------------------------------------------------------------
# TODO: 1) build the pipeline in CLIENT mode, so it runs on the API and reuses the
#          key from section 0:  TabPFNTSPipeline(tabpfn_mode=TabPFNMode.CLIENT)
#       2) forecast HORIZON months from history_df, asking for the 10th, 50th and
#          90th percentiles -> `forecast`
#          hint: pipeline.predict_df(history_df, prediction_length=..., quantiles=[...])
# your code here

# "target" is the point forecast; the other columns are the quantile levels you asked
# for. One row per forecast month, indexed by (item_id, timestamp).
forecast.head(3)


In [ ]:
from tabpfn_time_series.plot import plot_forecast

# Their plotting helper, written for exactly these DataFrames: it draws the context,
# the forecast, the quantile band and the ground truth.
plot_forecast(history_df, forecast, test_df=future_df, context_length=60)

actual = future_df["target"].to_numpy()

# The baseline every forecaster starts with: assume next year looks like last year.
last_year = history_df["target"].to_numpy()[-12:]
seasonal_naive = np.resize(last_year, HORIZON)

print(f"TabPFN-TS         RMSE {root_mean_squared_error(actual, forecast['target']):.2f} ppm")
print(f"repeat last year  RMSE {root_mean_squared_error(actual, seasonal_naive):.2f} ppm")


The docs: [Forecasting](https://docs.priorlabs.ai/capabilities/forecasting?utm_source=workshop&utm_campaign=pyladies).

---
## 5. Advanced — spending more compute for a better model

Everything so far has been a single forward pass. **Thinking mode** is the tabular
version of the idea you already know from LLMs: let the model spend more compute
before it answers, and get a better answer.

Instead of one pass with default settings, TabPFN-3-Plus searches over configurations
at fit time and keeps whatever scores best on **the metric you name**. You pay once,
at fit time; every prediction afterwards is as cheap as before. Worth up to **15%
better on TabArena Elo** than vanilla TabPFN-3, per the
[technical report](https://arxiv.org/abs/2605.13986).

```python
TabPFNClassifier(
    thinking_mode=True,
    thinking_effort="medium",   # or "high"
    thinking_metric="roc_auc",  # optimise what you actually care about
    thinking_timeout_s=180,     # your compute budget, in seconds
)
```

`thinking_metric` is the part worth pausing on. If your real problem is "catch as many
malignant cases as possible", you can optimise `recall` instead of accuracy and the
search will chase that. No custom training loop, one keyword argument.

Two practical notes: it is **API-only** (not in the open-source package), and thinking
fits draw on a **separate monthly quota** (20 by default). So the cell below is **off
by default** — flip the flag if you have quota to spare. It also takes a couple of
minutes, which is why we are not asking forty people to run it at once.

In [ ]:
RUN_THINKING = False  # set to True to spend one thinking fit from your monthly quota

if RUN_THINKING:
    started = time.perf_counter()
    clf_thinking = TabPFNClassifier(
        thinking_mode=True,
        thinking_effort="medium",   # or "high"
        thinking_metric="roc_auc",  # optimise the metric you actually care about
        thinking_timeout_s=180,
    )
    clf_thinking.fit(context_features, context_labels)
    thinking_proba = clf_thinking.predict_proba(query_features)[:, 1]

    print(f"vanilla TabPFN-3 ROC AUC: {roc_auc_score(query_labels, proba):.4f}")
    print(f"thinking mode    ROC AUC: {roc_auc_score(query_labels, thinking_proba):.4f}")
    print(f"fit time: {time.perf_counter() - started:.0f}s")
else:
    print("Skipped. See https://docs.priorlabs.ai/capabilities/thinking-mode?utm_source=workshop&utm_campaign=pyladies")

---
## Predictive Agents

One last thing, because it is where this is all heading.

An LLM agent is good at reading, writing and deciding what to do next. It is bad at
"how many units will this SKU sell next month" — the thing you just did twice in this
notebook. Those are complementary weaknesses, so the obvious move is to let the agent
*call* a model like TabPFN when it needs a number, the same way it calls a search tool
when it needs a fact.

Prior Labs ships an **MCP server** for exactly that: it exposes TabPFN's fit and
predict as tools any MCP-speaking agent (Claude Code, Claude Desktop, Cursor, your own)
can call. The agent brings the reasoning and the messy real-world context; TabPFN
brings the calibrated number.

**→ [docs.priorlabs.ai/agentic/mcp](https://docs.priorlabs.ai/agentic/mcp?utm_source=workshop&utm_campaign=pyladies)**

That is the subject of the follow-up workshop, so consider this a teaser rather than an
exercise.

---
### Keep going

- Your API keys and usage → [ux.priorlabs.ai](https://ux.priorlabs.ai/?utm_source=workshop&utm_campaign=pyladies)
- Docs, cookbook, thinking mode → [docs.priorlabs.ai](https://docs.priorlabs.ai/?utm_source=workshop&utm_campaign=pyladies)
- Open-source package (run it on your own GPU) → [github.com/PriorLabs/TabPFN](https://github.com/PriorLabs/TabPFN)
- API client → [github.com/PriorLabs/tabpfn-client](https://github.com/PriorLabs/tabpfn-client)
- TabPFN-3 technical report → [arxiv.org/abs/2605.13986](https://arxiv.org/abs/2605.13986)
- Community → [Prior Labs Discord](https://discord.gg/BHnX2Ptf4j)

### If you are keen

1. **Bring your own CSV.** Swap it into Exercise 1 — the code does not change. If your
   target is a number rather than a category, use Exercise 2 instead.
2. **Try a harder dataset.** The official demo uses Parkinson's voice measurements
   (195 rows, 22 features), where the baselines have much more trouble. Good way to
   see the gap this notebook's easy dataset hides.
3. Rerun the context sweep with `n_estimators=4` vs. the default 8 and see what
   ensembling over "prompts" buys you.
4. TabPFN-3 handles **text columns natively** through the API. Take a dataset with a
   free-text column, pass it straight in, and compare against your usual TF-IDF pipeline.

In [ ]:
print(tabpfn_client.get_api_usage())